<a href="https://colab.research.google.com/github/Kintsukuro1/Mineria-Datos/blob/main/proyecto_mineria_felipe_ruiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Base - Evaluación de Minería de Datos

## Información del Grupo

- **Nombre o Número de Grupo:** Grupo 1
- **Sección:** BIY7121-004D
- **Fecha de Entrega:** [Indicar la fecha de entrega del informe]

---

## Integrantes del Grupo

### Estudiante 1
- **Nombre:** Felipe
- **Apellido:** Ruiz
- **Correo institucional:** fe.ruizm@duocuc.cl

---

## Tema del Proyecto
Análisis y predicción del comportamiento de juegos de casino online a partir de variables como RTP, volatilidad, multiplicadores y características de los juegos.

> Clasificación de juegos de casino online según su potencial de retorno y multiplicador máximo, en base a variables técnicas y comerciales del dataset.

---

## Descripción del Proyecto
Este proyecto analiza un dataset de juegos de casino online con el fin de identificar patrones en las características de los juegos que permitan segmentarlos por atractivo para el jugador. El modelo resultante podría ser utilizado por operadores de casinos para personalizar su oferta y optimizar la retención de usuarios.

---

## Introducción a la Metodología

Este documento sigue la estructura de la metodología **CRISP-DM** (*Cross Industry Standard Process for Data Mining*), compuesta por seis fases fundamentales que guían el análisis de datos desde la comprensión del negocio hasta la evaluación del modelo.

---


## 1. Comprensión del Negocio

El objetivo del proyecto es analizar un dataset de juegos de casino online utilizando técnicas de minería de datos para identificar patrones comerciales, factores asociados a los ingresos y preferencias de los usuarios.

### Objetivo General
Analizar el comportamiento y rendimiento de casinos online mediante modelos de minería de datos para apoyar la toma de decisiones comerciales y estratégicas.

### Objetivos Específicos

1. **Identificar las variables que influyen en la recaudación de los casinos** utilizando un modelo de Árbol de Decisión.

2. **Predecir ingresos o rentabilidad de juegos/casinos** mediante un modelo de Regresión Múltiple.

3. **Descubrir patrones frecuentes de uso entre juegos** utilizando Market Basket Analysis.

### Modelos Seleccionados

| Modelo | Objetivo |
|---|---|
| Árbol de Decisión | Explicar qué factores influyen en mayores ingresos |
| Regresión Múltiple | Predecir ingresos o rentabilidad |
| Market Basket Analysis | Detectar asociaciones frecuentes entre juegos |

### Preguntas de Negocio

- ¿Qué casinos presentan mayor nivel de recaudación?
- ¿Qué juegos son más populares entre los usuarios?
- ¿Qué juegos generan mayor rentabilidad?
- ¿Qué patrones de juegos suelen repetirse entre los jugadores?


In [ ]:
# No requiere código en esta fase

## 2. Comprensión de los Datos

- **Fuente del dataset:** `online_casino_games_dataset_v2.csv`
- **Variables objetivo:** `rtp` (retorno al jugador) y `max_multiplier` (multiplicador máximo)
- **Variables predictoras:** casino, proveedor, tipo de juego, volatilidad, jackpot, apuesta mínima, ganancia máxima, compatibilidad móvil, free spins, compra de bonus, año de lanzamiento, jurisdicción, moneda.
- **Tipos de datos:** numéricos continuos, categóricos nominales/ordinales, booleanos, temporales.

A continuación se carga el dataset y se inspecciona su estructura inicial.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset
df = pd.read_csv('/content/online_casino_games_dataset_v2.csv')

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()


In [ ]:
# Inspección estructural: tipos de datos y conteo de no-nulos
df.info()


In [ ]:
# Nombres de columnas disponibles
print(df.columns.tolist())


In [ ]:
# Diccionario de datos: mapeo de columnas
diccionario_columnas = {
    "casino":               ("casino",                  "Nombre del casino",                        "Categórica", "Nominal"),
    "game":                 ("juego",                   "Nombre del juego",                         "Categórica", "Nominal"),
    "provider":             ("proveedor",               "Proveedor del juego",                      "Categórica", "Nominal"),
    "rtp":                  ("retorno_al_jugador",      "Porcentaje de retorno esperado",            "Numérica",   "Continua"),
    "volatility":           ("volatilidad",             "Nivel de volatilidad del juego",            "Categórica", "Ordinal"),
    "jackpot":              ("tiene_jackpot",           "Si el juego incluye jackpot",               "Categórica", "Binaria"),
    "country_availability": ("paises_disponibles",      "Países donde está habilitado",              "Categórica", "Nominal"),
    "min_bet":              ("apuesta_minima",          "Monto mínimo para apostar",                 "Numérica",   "Continua"),
    "max_win":              ("ganancia_maxima",         "Ganancia máxima potencial",                 "Numérica",   "Continua"),
    "game_type":            ("tipo_juego",              "Tipo general del juego",                    "Categórica", "Nominal"),
    "game_category":        ("categoria_juego",         "Categoría específica del juego",            "Categórica", "Nominal"),
    "license_jurisdiction": ("jurisdiccion_licencia",   "Jurisdicción regulatoria",                  "Categórica", "Nominal"),
    "release_year":         ("anio_lanzamiento",        "Año de lanzamiento",                        "Numérica",   "Discreta"),
    "currency":             ("moneda",                  "Moneda principal del juego",                "Categórica", "Nominal"),
    "mobile_compatible":    ("compatible_movil",        "Compatibilidad con móviles",                "Categórica", "Binaria"),
    "free_spins_feature":   ("tiene_tiros_gratis",      "Si tiene función de free spins",            "Categórica", "Binaria"),
    "bonus_buy_available":  ("compra_bonus_disponible", "Permite comprar bono directamente",         "Categórica", "Binaria"),
    "max_multiplier":       ("multiplicador_maximo",    "Máximo multiplicador de ganancia",          "Numérica",   "Discreta"),
    "languages":            ("idiomas",                 "Idiomas disponibles",                       "Categórica", "Nominal"),
    "last_updated":         ("fecha_actualizacion",     "Fecha de última actualización",             "Temporal",   "Fecha"),
}

filas = []
for col, (es, desc, tipo, subtipo) in diccionario_columnas.items():
    filas.append({
        "Columna original": col,
        "Nombre en español": es,
        "Descripción": desc,
        "Tipo general": tipo,
        "Subtipo": subtipo,
        "Dtype pandas": str(df[col].dtype) if col in df.columns else "No encontrada",
        "Existe": col in df.columns,
    })

pd.DataFrame(filas)


**Interpretación:** El dataset contiene 20 variables que cubren aspectos técnicos, comerciales y regulatorios de los juegos. Las variables numéricas continuas (`rtp`, `min_bet`, `max_win`) son candidatas a escalado. Las categóricas nominales requieren codificación. Las booleanas pueden tratarse como indicadores 0/1.


## 3. Preparación de los Datos

Esta fase incluye:
- Detección y limpieza de valores nulos
- Eliminación de duplicados
- Segmentación por tipo de variable
- Imputación diferenciada
- Encoding de categóricas
- Escalado de numéricas


In [ ]:
# ── Limpieza previa ──────────────────────────────────────────────────────────
# Eliminar duplicados
n_antes = len(df)
df = df.drop_duplicates()
print(f"Filas eliminadas por duplicados: {n_antes - len(df)}")

# Eliminar columnas de alta cardinalidad (no aptas para modelos directamente)
cols_alta_cardinalidad = ['game', 'languages', 'country_availability', 'provider', 'last_updated']
df = df.drop(columns=[c for c in cols_alta_cardinalidad if c in df.columns])
print(f"Columnas eliminadas: {cols_alta_cardinalidad}")
print(f"Shape resultante: {df.shape}")


### 3.1 Depuración de Datos

In [ ]:
# ── Diagnóstico de nulos ANTES de imputar ────────────────────────────────────
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': nulos_pct
}).query('Nulos > 0').sort_values('Nulos', ascending=False)

print("Columnas con valores nulos:")
print(resumen_nulos if not resumen_nulos.empty else "  → No hay valores nulos")


In [ ]:
# ── Segmentación por tipo de variable ────────────────────────────────────────
num_cols  = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols  = df.select_dtypes(include=['object']).columns.tolist()
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()

print(f"Numéricas  ({len(num_cols)}):  {num_cols}")
print(f"Categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Booleanas  ({len(bool_cols)}): {bool_cols}")


In [ ]:
# ── Imputación diferenciada ───────────────────────────────────────────────────
# Numéricas → mediana (robusta ante outliers)
for col in num_cols:
    mediana = df[col].median()
    n_imputados = df[col].isnull().sum()
    if n_imputados > 0:
        df[col] = df[col].fillna(mediana)
        print(f"  [{col}] {n_imputados} nulos → mediana = {mediana:.4f}")

# Categóricas → moda (categoría más frecuente)
for col in cat_cols:
    moda = df[col].mode(dropna=True)
    n_imputados = df[col].isnull().sum()
    if not moda.empty and n_imputados > 0:
        df[col] = df[col].fillna(moda.iloc[0])
        print(f"  [{col}] {n_imputados} nulos → moda = '{moda.iloc[0]}'")

# Booleanas → False (criterio conservador: sin evidencia positiva)
for col in bool_cols:
    n_imputados = df[col].isnull().sum()
    if n_imputados > 0:
        df[col] = df[col].fillna(False)
        print(f"  [{col}] {n_imputados} nulos → False")

print("\n✓ Imputación completada")


In [ ]:
# ── Verificación post-imputación ─────────────────────────────────────────────
nulos_restantes = df.isnull().sum().sum()
print(f"Nulos restantes en el dataset: {nulos_restantes}")
assert nulos_restantes == 0, "¡Aún quedan nulos sin imputar!"
print("✓ Dataset libre de valores nulos")


**Interpretación de la imputación:**
- **Numéricas → mediana**: reduce el impacto de outliers y preserva la distribución central.
- **Categóricas → moda**: conserva la categoría más representativa del dataset.
- **Booleanas → False**: criterio conservador cuando no existe evidencia positiva del atributo.


### 3.2 Encoding

In [ ]:
# ── One-Hot Encoding para categóricas de baja cardinalidad ──────────────────
# Actualizar lista de categóricas tras eliminar columnas
cat_cols_actuales = df.select_dtypes(include=['object']).columns.tolist()

# Solo columnas con menos de 10 categorías únicas (evita explosión dimensional)
cat_cols_ohe = [col for col in cat_cols_actuales if df[col].nunique() < 10]
cat_cols_label = [col for col in cat_cols_actuales if df[col].nunique() >= 10]

print(f"Columnas para OHE ({len(cat_cols_ohe)}):        {cat_cols_ohe}")
print(f"Columnas descartadas/alta cardinalidad ({len(cat_cols_label)}): {cat_cols_label}")

df_encoded = pd.get_dummies(df, columns=cat_cols_ohe, drop_first=True)

# Eliminar columnas de alta cardinalidad remanentes (no codificables eficientemente)
df_encoded = df_encoded.drop(columns=cat_cols_label, errors='ignore')

print(f"\nShape tras encoding: {df_encoded.shape}")


**Interpretación:** Se aplicó One-Hot Encoding a variables categóricas con menos de 10 categorías para evitar explosión de dimensionalidad. El parámetro `drop_first=True` reduce redundancia entre columnas codificadas.


In [ ]:
# ── Escalado de variables numéricas ──────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

# Identificar columnas numéricas en el dataset codificado
num_cols_encoded = df_encoded.select_dtypes(include=['int64', 'float64']).columns.tolist()

scaler = StandardScaler()
df_encoded[num_cols_encoded] = scaler.fit_transform(df_encoded[num_cols_encoded])

print(f"Variables escaladas ({len(num_cols_encoded)}): {num_cols_encoded}")
print("\nEstadísticas post-escalado (deben ser ≈ media=0, std=1):")
print(df_encoded[num_cols_encoded].describe().loc[['mean','std']].round(4))


In [ ]:
# ── Dataset final preparado ──────────────────────────────────────────────────
df_final = df_encoded.copy()

print(f"Shape final: {df_final.shape}")
print(f"Nulos: {df_final.isnull().sum().sum()}")
df_final.head()


In [ ]:
# Verificación final
df_final.info()


## 4. Análisis Exploratorio de Datos

Se exploran distribuciones, correlaciones y relaciones entre variables clave para identificar hipótesis relevantes antes del modelado.


In [ ]:
# ── Estadísticas descriptivas ────────────────────────────────────────────────
df_final.select_dtypes(include=[np.number]).describe().T.round(3)


In [ ]:
# ── Medidas de tendencia central ─────────────────────────────────────────────
num_df = df_final.select_dtypes(include=[np.number])

print("Media:")
print(num_df.mean().round(4))
print("\nMediana:")
print(num_df.median().round(4))


In [ ]:
# ── Dispersión: desviación estándar e IQR ────────────────────────────────────
print("Desviación estándar:")
print(num_df.std().round(4))

Q1 = num_df.quantile(0.25)
Q3 = num_df.quantile(0.75)
IQR = Q3 - Q1
print("\nIQR (Rango Intercuartílico):")
print(IQR.round(4))


In [ ]:
# ── Boxplot de variables numéricas originales ────────────────────────────────
# Usamos el df original (antes de escalar) para mejor interpretabilidad visual
vars_box = ['min_bet', 'max_win', 'rtp']
vars_box_presentes = [v for v in vars_box if v in df.columns]

fig, axes = plt.subplots(1, len(vars_box_presentes), figsize=(12, 5))
for ax, col in zip(axes, vars_box_presentes):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col)
    ax.set_ylabel('Valor')

plt.suptitle('Distribución y outliers - Variables numéricas clave', fontsize=13)
plt.tight_layout()
plt.show()


**Interpretación:** Los boxplots revelan la presencia de valores atípicos en `max_win` y `min_bet`, evidenciando juegos con características extremas. El `rtp` se muestra más concentrado y simétrico.


In [ ]:
# ── Matriz de correlación ────────────────────────────────────────────────────
corr_df = df_final.select_dtypes(include=[np.number])
corr_matrix = corr_df.corr()

plt.figure(figsize=(13, 9))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False,
            linewidths=0.3, square=True)
plt.title('Matriz de correlación entre variables numéricas', fontsize=13)
plt.tight_layout()
plt.show()

# Top 10 pares con mayor correlación absoluta
corr_pares = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
print("Top 10 pares con mayor correlación absoluta:")
print(corr_pares.head(10).round(4))


**Interpretación:** Correlaciones cercanas a ±1 indican relación lineal fuerte. Pares con alta correlación entre predictores pueden indicar redundancia, lo que conviene revisar antes del modelado para evitar multicolinealidad.


## 5. Modelado y Justificación

### a. Enfoque del proyecto

El proyecto utiliza tres modelos de minería de datos con objetivos distintos para responder preguntas de negocio relevantes dentro del contexto de casinos online.

### b. Modelos Implementados

#### 1. Árbol de Decisión
Se utilizará para identificar las variables que más influyen en los ingresos o rendimiento de los casinos.

#### 2. Regresión Múltiple
Permitirá predecir la rentabilidad o ingresos de un casino en función de múltiples variables operacionales.

#### 3. Market Basket Analysis
Permitirá descubrir asociaciones frecuentes entre juegos utilizados por los jugadores.

### c. Justificación de selección

Los modelos fueron seleccionados debido a que combinan:
- Interpretabilidad
- Capacidad predictiva
- Descubrimiento de patrones
- Aplicación comercial real


In [ ]:
# ─────────────────────────────────────────────────────────────
# Preparación de variables para modelado
# ─────────────────────────────────────────────────────────────

# Crear variable aproximada de ingresos/rentabilidad
# (puede ajustarse según el dataset disponible)

df_model = df.copy()

# Variable simulada basada en max_win y RTP
df_model['ingresos_estimados'] = (
    df_model['max_win'].fillna(df_model['max_win'].median()) *
    (100 - df_model['rtp'].fillna(df_model['rtp'].median()))
)

# Clasificación binaria para Árbol de Decisión
mediana_ingresos = df_model['ingresos_estimados'].median()

df_model['alto_ingreso'] = (
    df_model['ingresos_estimados'] > mediana_ingresos
).astype(int)

print(df_model[['ingresos_estimados', 'alto_ingreso']].head())


### c. Modelos utilizados

1. **Árbol de Decisión**
   - Explica factores asociados a altos ingresos.
   - Modelo interpretable y visual.

2. **Regresión Múltiple**
   - Predice ingresos estimados.
   - Permite identificar influencia de múltiples variables.

3. **Market Basket Analysis**
   - Descubre relaciones frecuentes entre juegos.
   - Útil para recomendaciones y estrategias comerciales.


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Codificación de variables categóricas
df_encoded_model = df_model.copy()

for col in df_encoded_model.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_encoded_model[col] = le.fit_transform(df_encoded_model[col].astype(str))

# Variables predictoras
X = df_encoded_model.drop(columns=['alto_ingreso', 'ingresos_estimados'])

# Variables objetivo
y_clasificacion = df_encoded_model['alto_ingreso']
y_regresion = df_encoded_model['ingresos_estimados']

print(X.shape)


In [ ]:
from sklearn.model_selection import train_test_split

# División para clasificación
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clasificacion,
    test_size=0.2,
    random_state=42
)

# División para regresión
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_regresion,
    test_size=0.2,
    random_state=42
)

print("Datos preparados correctamente")


## 6. Entrenamiento y Evaluación de Modelos

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# ─────────────────────────────────────────────────────────────
# MODELO 1: Árbol de Decisión
# ─────────────────────────────────────────────────────────────

tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(X_train_c, y_train_c)

pred_tree = tree_model.predict(X_test_c)

print("Accuracy Árbol de Decisión:")
print(accuracy_score(y_test_c, pred_tree))

print("\nReporte de clasificación:")
print(classification_report(y_test_c, pred_tree))


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# ─────────────────────────────────────────────────────────────
# MODELO 2: Regresión Múltiple
# ─────────────────────────────────────────────────────────────

reg_model = LinearRegression()

reg_model.fit(X_train_r, y_train_r)

pred_reg = reg_model.predict(X_test_r)

print("R2 Score:")
print(r2_score(y_test_r, pred_reg))

print("\nMAE:")
print(mean_absolute_error(y_test_r, pred_reg))


In [ ]:
# ─────────────────────────────────────────────────────────────
# MODELO 3: Market Basket Analysis (simplificado)
# ─────────────────────────────────────────────────────────────

# Ejemplo básico de frecuencia de juegos por casino

basket = pd.crosstab(df['casino'], df['game_type'])

print("Matriz de asociaciones:")
print(basket.head())

# Juegos más frecuentes
print("\nJuegos más utilizados:")
print(df['game_type'].value_counts().head())


**Interpretación de resultados**

- El Árbol de Decisión permite identificar qué variables tienen mayor impacto sobre los ingresos.
- La Regresión Múltiple ayuda a estimar ingresos utilizando múltiples variables predictoras.
- El Market Basket Analysis permite descubrir patrones frecuentes entre tipos de juegos y preferencias de usuarios.

Estos modelos permiten combinar análisis descriptivo, predictivo y de asociaciones dentro de un mismo proyecto de minería de datos.


## 7. Despliegue (Opcional)

Los modelos podrían implementarse dentro de una plataforma de casino online para:

- Recomendar juegos populares.
- Detectar juegos altamente rentables.
- Identificar patrones de comportamiento de usuarios.
- Mejorar estrategias de marketing y promociones.
- Apoyar decisiones comerciales basadas en datos.


In [ ]:
# ── Simulación de predicción para un nuevo juego ─────────────────────────────
# Ejemplo: usar el mejor modelo para predecir sobre los primeros 5 registros de test
muestra = X_test.iloc[:5]
preds = tree_model.predict(muestra)

print("Predicciones para 5 juegos de ejemplo:")
for i, pred in enumerate(preds):
    etiqueta = "Alto potencial" if pred == 1 else "Estándar"
    print(f"  Juego {i+1}: {etiqueta}")


## 8. Conclusiones y Recomendaciones

### Hallazgos Esperados

- Existen variables que influyen directamente en los ingresos de los casinos.
- Algunos juegos presentan mayor popularidad y rentabilidad.
- Los usuarios tienden a repetir combinaciones frecuentes de juegos.

### Recomendaciones

- Utilizar modelos predictivos para apoyar decisiones comerciales.
- Priorizar juegos con alta rentabilidad y popularidad.
- Implementar sistemas de recomendación basados en asociaciones frecuentes.
- Continuar ampliando el análisis con nuevos modelos y métricas.

### Conclusión General

La minería de datos permite transformar información operacional de casinos online en conocimiento útil para optimizar estrategias comerciales, mejorar la experiencia de usuario y apoyar la toma de decisiones.
